In [19]:
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.subplots import make_subplots
import fastf1
import numpy as np
import pandas as pd

In [20]:
session_monza_q = fastf1.get_session(2023, 'Monza', 'Q')
session_monza_q.load()

session_bahrain_r = fastf1.get_session(2023, 'Bahrain', 'R')
session_bahrain_r.load()

lap_SAI = session_monza_q.laps.pick_drivers('SAI').pick_fastest()
lap_STR = session_monza_q.laps.pick_drivers('STR').pick_fastest()

tel_SAI = lap_SAI.get_telemetry()
tel_STR = lap_STR.get_telemetry()

laps_VER = session_bahrain_r.laps.pick_drivers('VER')
laps_ALO = session_bahrain_r.laps.pick_drivers('ALO')

Request for URL https://raw.githubusercontent.com/theOehrly/f1schedule/master/schedule_2023.json failed; using cached response
Traceback (most recent call last):
  File "/Users/edouardbougeant/motorsport_env/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/Users/edouardbougeant/motorsport_env/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/Users/edouardbougeant/motorsport_env/lib/python3.12/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1450, in getresponse
    response.begin()
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Framework

In [21]:
from scipy.interpolate import interp1d

f_SAI = interp1d(tel_SAI['Distance'].values,
                 tel_SAI['Time'].dt.total_seconds().values,
                 bounds_error=False, fill_value='extrapolate')

f_STR = interp1d(tel_STR['Distance'].values,
                 tel_STR['Time'].dt.total_seconds().values,
                 bounds_error=False, fill_value='extrapolate')

dist_common = np.linspace(0, 5793, 500)

delta = f_SAI(dist_common) - f_STR(dist_common)


fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=dist_common,
    y=delta,
    mode='lines',
    name='SAI - STR',
    line=dict(color='purple', width=2)
))
fig1.add_hline(y=0, line_dash='dash', line_color='white')
fig1.update_layout(
    title='Time Delta — SAI vs STR Monza 2023 Qualifying',
    xaxis_title='Distance (m)',
    yaxis_title='Time Difference (s)',
    template='plotly_dark',
    height=500
)
fig1.write_html("dashboard_time_delta.html")

In [22]:
scenarios = ['+10% Cl', '-10% Cx', '+100kW power', '-50kg mass']
deltas    = [0.591, 0.690, 1.810, 0.911]
colors    = ['blue', 'orange', 'green', 'red']

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=scenarios,
    y=deltas,
    marker_color=colors,
    text=[f'+{d:.3f}s' for d in deltas],
    textposition='outside'
))

fig2.update_layout(
    title='Performance Gain vs Baseline — Monza 2023 LTS',
    xaxis_title='Scenario',
    yaxis_title='Time gained (s)',
    template='plotly_dark',
    height=500
)

fig2.write_html("dashboard_lts_scenarios.html")

In [23]:
FUEL_EFFECT = 2.3 * 0.03

lap_times_VER = laps_VER['LapTime'].dt.total_seconds()
median_VER    = lap_times_VER.median()
laps_VER_clean = laps_VER[
    (lap_times_VER < median_VER * 1.03) &
    (lap_times_VER > median_VER * 0.97)
].copy()

lap_times_ALO = laps_ALO['LapTime'].dt.total_seconds()
median_ALO    = lap_times_ALO.median()
laps_ALO_clean = laps_ALO[
    (lap_times_ALO < median_ALO * 1.03) &
    (lap_times_ALO > median_ALO * 0.97)
].copy()

laps_VER_clean['LapTime_corrected'] = (
    laps_VER_clean['LapTime'].dt.total_seconds() +
    FUEL_EFFECT * (laps_VER_clean['LapNumber'] - 1))

laps_ALO_clean['LapTime_corrected'] = (
    laps_ALO_clean['LapTime'].dt.total_seconds() +
    FUEL_EFFECT * (laps_ALO_clean['LapNumber'] - 1))

stint_colors = {1: 'blue', 2: 'orange', 3: 'green'}

fig3 = make_subplots(rows=1, cols=2,
                     subplot_titles=['Verstappen', 'Alonso'])

for stint in laps_VER_clean['Stint'].unique():
    d = laps_VER_clean[laps_VER_clean['Stint'] == stint]
    compound = d['Compound'].iloc[0]
    fig3.add_trace(go.Scatter(
        x=d['TyreLife'],
        y=d['LapTime_corrected'],
        mode='markers',
        name=f'VER Stint {int(stint)} {compound}',
        marker=dict(color=stint_colors[int(stint)], size=8)
    ), row=1, col=1)

for stint in laps_ALO_clean['Stint'].unique():
    d = laps_ALO_clean[laps_ALO_clean['Stint'] == stint]
    compound = d['Compound'].iloc[0]
    fig3.add_trace(go.Scatter(
        x=d['TyreLife'],
        y=d['LapTime_corrected'],
        mode='markers',
        name=f'ALO Stint {int(stint)} {compound}',
        marker=dict(color=stint_colors[int(stint)],
                    size=8, symbol='diamond')
    ), row=1, col=2)

fig3.update_layout(
    title='Tyre Degradation — VER vs ALO Bahrain 2023 Race',
    template='plotly_dark',
    height=500
)
fig3.update_xaxes(title_text='Tyre Life (laps)')
fig3.update_yaxes(title_text='Corrected Lap Time (s)')

for stint in laps_VER_clean['Stint'].unique():
    d        = laps_VER_clean[laps_VER_clean['Stint'] == stint]
    x        = d['TyreLife'].values
    y        = d['LapTime_corrected'].values
    coeffs   = np.polyfit(x, y, 1)
    trend    = np.poly1d(coeffs)
    compound = d['Compound'].iloc[0]

    fig3.add_trace(go.Scatter(
        x=x,
        y=trend(x),
        mode='lines',
        name=f'VER Stint {int(stint)} trend',
        line=dict(color=stint_colors[int(stint)],
                  dash='dash', width=1.5),
        showlegend=False
    ), row=1, col=1)

for stint in laps_ALO_clean['Stint'].unique():
    d        = laps_ALO_clean[laps_ALO_clean['Stint'] == stint]
    x        = d['TyreLife'].values
    y        = d['LapTime_corrected'].values
    coeffs   = np.polyfit(x, y, 1)
    trend    = np.poly1d(coeffs)

    fig3.add_trace(go.Scatter(
        x=x,
        y=trend(x),
        mode='lines',
        name=f'ALO Stint {int(stint)} trend',
        line=dict(color=stint_colors[int(stint)],
                  dash='dash', width=1.5),
        showlegend=False
    ), row=1, col=2)

fig3.write_html("dashboard_tyre_degradation.html")

In [24]:
from scipy.interpolate import interp1d

dist_common = np.linspace(0, 5793, 1000)

channels = ['Speed', 'Throttle', 'Brake', 'nGear', 'RPM']
units    = ['km/h', '%', '0/1', 'Gear', 'RPM']

fig4 = make_subplots(rows=5, cols=1,
                     shared_xaxes=True,
                     subplot_titles=channels,
                     vertical_spacing=0.05)

for i, (ch, unit) in enumerate(zip(channels, units)):

    f_SAI = interp1d(tel_SAI['Distance'].values,
                     tel_SAI[ch].values,
                     bounds_error=False,
                     fill_value='extrapolate')
 
    f_STR = interp1d(tel_STR['Distance'].values,
                     tel_STR[ch].values,
                     bounds_error=False,
                     fill_value='extrapolate')

    fig4.add_trace(go.Scatter(
        x=dist_common,
        y=f_SAI(dist_common),
        mode='lines',
        name='SAI' if i == 0 else 'SAI',
        line=dict(color='red', width=1.5),
        showlegend=(i == 0)
    ), row=i+1, col=1)

    fig4.add_trace(go.Scatter(
        x=dist_common,
        y=f_STR(dist_common),
        mode='lines',
        name='STR' if i == 0 else 'STR',
        line=dict(color='green', width=1.5),
        showlegend=(i == 0)
    ), row=i+1, col=1)

    fig4.update_yaxes(title_text=unit, row=i+1, col=1)

fig4.update_xaxes(title_text='Distance (m)', row=5, col=1)
fig4.update_layout(
    title='Multi-Channel Overlay — SAI vs STR Monza 2023 Q',
    template='plotly_dark',
    height=1000
)

fig4.write_html("dashboard_multichannel.html")

In [25]:
def simulate_acceleration_dash(Cx, d_total=750, v_start=280/3.6):
    P_max = 735000
    mass  = 798
    rho   = 1.225
    A     = 1.5
    Rx    = 0.005
    g     = 9.81
    v     = v_start
    distances = [0]
    speeds    = [v * 3.6]
    d         = 0
    while d < d_total:
        F_drag   = 0.5 * rho * Cx * A * v**2
        F_roll   = Rx * mass * g
        F_engine = min(P_max/v, mass*g*1.8)
        a        = (F_engine - F_drag - F_roll) / mass
        v       += a * 0.01
        d       += v * 0.01
        distances.append(d)
        speeds.append(v * 3.6)
    return distances, speeds

d_off, s_off = simulate_acceleration_dash(Cx=1.064)
d_on,  s_on  = simulate_acceleration_dash(Cx=0.866)

straight  = tel_SAI[tel_SAI['Distance'] < 750]
drs_on_s  = straight[straight['DRS'] >= 12]

fig5 = make_subplots(rows=1, cols=2,
                     subplot_titles=[
                         'DRS Effect — Main Straight Monza',
                         'Drag Coefficient Cx — Comparison'])

fig5.add_trace(go.Scatter(
    x=d_off, y=s_off,
    mode='lines', name='DRS OFF (theoretical)',
    line=dict(color='blue', width=2)
), row=1, col=1)

fig5.add_trace(go.Scatter(
    x=d_on, y=s_on,
    mode='lines', name='DRS ON (theoretical)',
    line=dict(color='red', width=2, dash='dash')
), row=1, col=1)

fig5.add_trace(go.Scatter(
    x=drs_on_s['Distance'].values,
    y=drs_on_s['Speed'].values,
    mode='lines', name='DRS ON (real FastF1)',
    line=dict(color='orange', width=1.5)
), row=1, col=1)

configs = ['Monza DRS OFF', 'Monza DRS ON',
           'Monaco DRS ON*', 'Monaco DRS OFF*',
           'WE25 Calibrated']
cx_vals = [1.064, 0.866, 1.524, 1.540, 0.694]
colors  = ['salmon', 'red', 'cornflowerblue',
           'darkblue', 'green']

fig5.add_trace(go.Bar(
    x=configs,
    y=cx_vals,
    marker_color=colors,
    text=[f'{c:.3f}' for c in cx_vals],
    textposition='outside',
    showlegend=False
), row=1, col=2)

fig5.update_layout(
    title='Aerodynamic Analysis — Monza vs Monaco 2023',
    template='plotly_dark',
    height=500
)
fig5.update_xaxes(title_text='Distance (m)', row=1, col=1)
fig5.update_yaxes(title_text='Speed (km/h)', row=1, col=1)
fig5.update_yaxes(title_text='Cx', row=1, col=2)

fig5.write_html("dashboard_aero.html")

In [26]:
fig1_html = fig1.to_html(full_html=False, include_plotlyjs='cdn')
fig2_html = fig2.to_html(full_html=False, include_plotlyjs=False)
fig3_html = fig3.to_html(full_html=False, include_plotlyjs=False)
fig4_html = fig4.to_html(full_html=False, include_plotlyjs=False)
fig5_html = fig5.to_html(full_html=False, include_plotlyjs=False)

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Motorsport Telemetry Dashboard</title>
    <style>
        body {{
            background-color: #111111;
            color: white;
            font-family: Arial, sans-serif;
            margin: 40px;
        }}
        h1 {{
            text-align: center;
            color: #FF1801;
            font-size: 2em;
            margin-bottom: 10px;
        }}
        h2 {{
            color: #aaaaaa;
            text-align: center;
            font-size: 1em;
            margin-bottom: 40px;
        }}
        .section {{
            margin-bottom: 60px;
            border-top: 1px solid #333;
            padding-top: 30px;
        }}
        .section-title {{
            color: #FF1801;
            font-size: 1.2em;
            margin-bottom: 10px;
        }}
    </style>
</head>
<body>
    <h1> Motorsport Telemetry Analysis</h1>
    <h2>F1 Data Analysis Portfolio  Edouard Bougeant</h2>

    <div class="section">
        <p class="section-title">01  Time Delta Analysis
        (SAI vs STR Monza 2023 Q)</p>
        {fig1_html}
    </div>

    <div class="section">
        <p class="section-title">02  Lap Time Simulation
        Scenario Analysis (Monza 2023)</p>
        {fig2_html}
    </div>

    <div class="section">
        <p class="section-title">03  Tyre Degradation
        (VER vs ALO Bahrain 2023 Race)</p>
        {fig3_html}
    </div>

    <div class="section">
        <p class="section-title">04  Multi-Channel Overlay
        (SAI vs STR Monza 2023 Q)</p>
        {fig4_html}
    </div>

    <div class="section">
        <p class="section-title">05  Aerodynamic Analysis
        (Monza vs Monaco 2023)</p>
        {fig5_html}
    </div>

</body>
</html>
"""

with open('motorsport_dashboard.html', 'w') as f:
    f.write(html_content)